# Traducción de Bitext EN → ES / PT con NLLB-200

Traduce el dataset `bitext_train.parquet` (27k customer support utterances en inglés)
a español y portugués usando el modelo **NLLB-200-distilled-600M** de Meta.

## Pipeline
1. Cargar bitext desde Google Drive
2. Traducir `instruction` EN → ES y guardar a Drive
3. Traducir `instruction` EN → PT y guardar a Drive

**Resultado:** ~27k ES + ~27k PT = ~54k muestras reales de soporte traducidas

⚠️ **Checkpoint automático:** cada idioma se guarda por separado. Si se cierra Colab,
solo repetís el idioma que faltaba.

In [ ]:
# @title 1. Montar Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Carpeta raíz del proyecto en Drive
DRIVE_DIR = 'content/drive/MyDrive/Datos-Drive/ConversaAI'
os.makedirs(f'{DRIVE_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/translated', exist_ok=True)

print(f'Drive montado en: {DRIVE_DIR}')
print('\n⚠️ IMPORTANTE: Subí bitext_train.parquet a:')
print(f'   {DRIVE_DIR}/data/raw/')
print('\n   Podés subirlo desde el panel de archivos de Colab → Drive.')

# Verificar qué hay
!ls -lh "$DRIVE_DIR/data/raw/" 2>/dev/null || echo '   (vacío — subí el archivo primero)'

In [ ]:
# @title 2. Verificar GPU
import torch, sys, pandas as pd, numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: Sin GPU. Activá Runtime > Change runtime type > T4 GPU")

!nvidia-smi

In [ ]:
# @title 3. Instalar dependencias
!pip install -q transformers datasets accelerate sentencepiece pandas numpy protobuf

In [ ]:
# @title 4. Cargar bitext desde Drive
import pyarrow.parquet as pq

src_path = f'{DRIVE_DIR}//content/drive/MyDrive/Datos-Drive/ConversaAI/data/raw/bitext_train.parquet'
if not os.path.exists(src_path):
    # Buscar alternativas
    alt = !ls "$DRIVE_DIR/data/raw/"
    print(f'ERROR: No se encontró bitext_train.parquet en {DRIVE_DIR}/data/raw/')
    print(f'Archivos encontrados: {alt}')
    raise FileNotFoundError(f'Subí bitext_train.parquet a {DRIVE_DIR}/data/raw/')

df = pq.read_table(src_path).to_pandas()
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nUnique intents: {df['intent'].nunique()}")
print(f"\nDistribución por categoría:")
print(df['category'].value_counts().to_string())
print(f"\nDistribución por intent:")
print(df['intent'].value_counts().to_string())
print(f"\nFlags disponibles: {df['flags'].unique()}")

df.head(3)

In [ ]:
# @title 5. Cargar NLLB-200
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/nllb-200-distilled-600M"
print(f"Cargando {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Modelo cargado en {device}")
print(f"Parámetros: {model.num_parameters():,}")

In [ ]:
# @title 6. Función de traducción batch (con checkpoint)
from tqdm.auto import tqdm
import json

BATCH_SIZE = 32
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

def translate_with_checkpoint(texts, target_lang, checkpoint_path, batch_size=BATCH_SIZE):
    """
    Traduce textos con checkpoint parcial.
    Si ya existe un checkpoint, pregunta si retomarlo.
    """
    start_idx = 0
    translations = []

    # Verificar checkpoint existente
    if os.path.exists(checkpoint_path):
        print(f"⚠️ Checkpoint encontrado: {checkpoint_path}")
        cp = torch.load(checkpoint_path)
        translations = cp['translations']
        start_idx = len(translations)
        print(f"   Retomando desde el índice {start_idx}/{len(texts)}")
    else:
        translations = [None] * len(texts)

    for i in tqdm(range(start_idx, len(texts), batch_size), desc=f"EN → {target_lang}", initial=start_idx//batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LENGTH
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.lang_code_to_id[target_lang],
                max_length=MAX_TARGET_LENGTH,
                num_beams=3
            )

        batch_translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        for j, t in enumerate(batch_translations):
            translations[i + j] = t

        # Checkpoint cada 5 batches
        if (i // batch_size) % 5 == 0 and i > start_idx:
            torch.save({'translations': translations, 'idx': i + batch_size}, checkpoint_path)
            torch.cuda.empty_cache()

    # Checkpoint final
    torch.save({'translations': translations, 'idx': len(texts)}, checkpoint_path)

    # Verificar que no haya None
    nones = sum(1 for t in translations if t is None)
    if nones > 0:
        print(f"⚠️ {nones} traducciones fallaron, rellenando con ''")
        translations = [t if t is not None else '' for t in translations]

    return translations

# Test rápido
test_texts = [
    "I need to cancel my order",
    "Where can I file a complaint?",
    "How do I reset my password?"
]

print("Test traducción EN → ES:")
es_test = translate_with_checkpoint(
    test_texts, "spa_Latn",
    f'{DRIVE_DIR}/data/translated/TEST_checkpoint.pt',
    batch_size=3
)
for en, es in zip(test_texts, es_test):
    print(f"  EN: {en}")
    print(f"  ES: {es}\n")
!rm "{DRIVE_DIR}/data/translated/TEST_checkpoint.pt" 2>/dev/null
print("✅ Test OK. Listo para traducir el dataset completo.")

In [ ]:
# @title 7. Traducir EN → ES (con checkpoint en Drive)
texts_to_translate = df['instruction'].tolist()
checkpoint_es = f'{DRIVE_DIR}/data/translated/checkpoint_es.pt'

print(f"Traduciendo {len(texts_to_translate)} textos EN → ES...")
print(f"Checkpoint → {checkpoint_es}")

translations_es = translate_with_checkpoint(texts_to_translate, "spa_Latn", checkpoint_es)

df_es = df[['intent', 'category']].copy()
df_es['text'] = translations_es
df_es['lang'] = 'es'

# Guardar a Drive inmediatamente
es_drive = f'{DRIVE_DIR}/data/translated/bitext_intent_es.csv'
df_es.to_csv(es_drive, index=False)

print(f"\n✅ ES guardado en Drive: {es_drive}")
print(f"   Muestras: {len(df_es)}")
print(f"\nMuestras:")
for i in range(5):
    print(f"  [{df.iloc[i]['intent']}] EN: {df.iloc[i]['instruction'][:80]}")
    print(f"  → ES: {df_es.iloc[i]['text'][:80]}\n")

In [ ]:
# @title 8. Traducir EN → PT (con checkpoint en Drive)
checkpoint_pt = f'{DRIVE_DIR}/data/translated/checkpoint_pt.pt'

print(f"Traduciendo {len(texts_to_translate)} textos EN → PT...")
print(f"Checkpoint → {checkpoint_pt}")

translations_pt = translate_with_checkpoint(texts_to_translate, "por_Latn", checkpoint_pt)

df_pt = df[['intent', 'category']].copy()
df_pt['text'] = translations_pt
df_pt['lang'] = 'pt'

# Guardar a Drive
pt_drive = f'{DRIVE_DIR}/data/translated/bitext_intent_pt.csv'
df_pt.to_csv(pt_drive, index=False)

print(f"\n✅ PT guardado en Drive: {pt_drive}")
print(f"   Muestras: {len(df_pt)}")
print(f"\nMuestras:")
for i in range(5):
    print(f"  [{df.iloc[i]['intent']}] EN: {df.iloc[i]['instruction'][:80]}")
    print(f"  → PT: {df_pt.iloc[i]['text'][:80]}\n")

In [ ]:
# @title 9. Verificar en Drive
print("Archivos en Drive:")
!ls -lh "$DRIVE_DIR/data/translated/"

# Limpiar checkpoints (ya completamos todo)
!rm -f "$DRIVE_DIR/data/translated/checkpoint_es.pt"
!rm -f "$DRIVE_DIR/data/translated/checkpoint_pt.pt"
print("\n✅ Checkpoints eliminados (traducción completa)")

In [ ]:
# @title 10. Validar calidad
def validate_translations(df, name):
    issues = []
    empty = df['text'].str.strip().str.len() < 5
    if empty.any():
        issues.append(f"{empty.sum()} textos vacíos/muy cortos")
    dups = df['text'].value_counts()
    high_dup = dups[dups > 20]
    if len(high_dup) > 0:
        issues.append(f"{len(high_dup)} textos repetidos >20 veces")
    if issues:
        print(f"⚠️ {name}:")
        for i in issues:
            print(f"   - {i}")
    else:
        print(f"✅ {name}: OK")

validate_translations(df_es, "ES")
validate_translations(df_pt, "PT")

---
## Resumen

| Archivo | En Drive | Filas |
|---------|----------|-------|
| `bitext_intent_es.csv` | ✅ | ~27k |
| `bitext_intent_pt.csv` | ✅ | ~27k |

**Todo persistió en Drive.** El próximo notebook (`colab_intent_finetune.ipynb`)
lee directo de `{DRIVE_DIR}/data/translated/`.